## About WindTrace
WindTrace is a python open-source parametric life-cycle inventory model to create tailor-made inventories of onshore wind turbines and park fleets and assess their environmental impacts. 

**IMPORTANT**: in order to run WindTrace you must have an Ecoinvent license. WindTrace supports multiple ecoinvent versions through a versioned activity selector CSV file. The package includes a default `ecoinvent_activity_variables.csv`, and users can extend or edit it for other ecoinvent versions using the updater utility.

More information on GitHub: https://github.com/LIVENlab/WindTrace_public

Copyright (c) 2025 LIVEN lab 
Licensed under the MIT License. See LICENSE for details.
This work has been funded by the European Commission though the Horizon Europe project "JUSTWIND4ALL" (GA 101083936). 

## WindTrace step-by-step

### Outlook

#### 1. Create environment and install WindTrace
#### 2. Prepare the WindTrace configuration file
#### 3. Load the Brightway project and databases
#### 3.1 Optional: import ecoinvent into the configured Brightway project
#### 4. Validate the ecoinvent activity selector CSV
#### 5. Create life-cycle inventories
##### 5.1 Example of application
##### 5.2 Exploring the inventories
##### 5.3 Calculating LCIA scores
##### 5.4 Another example with land use focus
##### 5.5 Creating a future wind park

### 1. Create environment and install WindTrace

From the package root folder, where `pyproject.toml` and `environment.yml` are located, run:

```bash
conda env create -f environment.yml
conda activate windtrace_env
pip install -e .
```

If WindTrace was installed from PyPI instead, activate your environment and run:
```bash
pip install windtrace
```
Then test:
```bash
python -c "import windtrace; print(windtrace.__version__)"
windtrace-ecoinvent-updater --help
```

### 2. Prepare the WindTrace configuration file

WindTrace uses `windtrace_config.yml` .

Before running this notebook:
1. Copy this notebook to your project folder.
1. Copy `windtrace_config.example.yml` to `windtrace_config.yml`and place it in your project folder.
2. Edit the Brightway project name, source ecoinvent database name, output database name, biosphere database name, ecoinvent version, and activity selector CSV path.
3. Copy the default `ecoinvent_activity_variables.csv` to your project folder if you want to edit it.

### 3. Load the Brightway project and databases
WindTrace reads the Brightway project name, ecoinvent database name, output database name, biosphere database name, ecoinvent version, and activity selector CSV path from `windtrace_config.yml`.

Edit `windtrace_config.yml` before running this notebook.

In [ ]:
from pathlib import Path
import os

import bw2data as bd
import bw2io as bi

from windtrace import load_config, apply_config, setup_brightway
from windtrace import lci_wind_turbine, lca_wind_turbine

In [ ]:
config = load_config("windtrace_config.yml")
apply_config(config)

bd.projects.set_current(config.project_name, update=False)

print("Brightway project:", config.project_name)
print("Source ecoinvent database:", config.source_database)
print("WindTrace output database:", config.output_database)
print("Biosphere database:", config.biosphere_database)
print("Ecoinvent version:", config.ecoinvent_version)
print("Activity variables CSV:", config.activity_variables_csv)
print("SPOLD files path:", config.spold_files)

### 3.1 Optional: import ecoinvent into the configured Brightway project

If the ecoinvent database listed in `windtrace_config.yml` already exists in your Brightway project, skip this section.

The imported database name must match:

```python
config.source_database
```

#### Import ecoinvent using your ecoinvent account

This code downloads/imports the ecoinvent release defined in `windtrace_config.yml`.

Before running this cell, set your ecoinvent license username and password in the cell.

In [ ]:
# Set this to True only if you want to run this import cell.
RUN_ONLINE_ECOINVENT_IMPORT = False

if RUN_ONLINE_ECOINVENT_IMPORT:
    # Replace these placeholders with your own ecoinvent credentials.
    # Do not save or share this notebook with real credentials inside it
    ecoinvent_username = "YOUR_ECOINVENT_USERNAME"
    ecoinvent_password = "YOUR_ECOINVENT_PASSWORD"

    if (
    not ecoinvent_username
    or not ecoinvent_password
    or ecoinvent_username == "YOUR_ECOINVENT_USERNAME"
    or ecoinvent_password == "YOUR_ECOINVENT_PASSWORD"
    ):
        raise ValueError(
        "Please replace YOUR_ECOINVENT_USERNAME and YOUR_ECOINVENT_PASSWORD "
        "with your own ecoinvent login credentials before running this cell."
        )


    system_model = "cutoff"

    expected_database_name = f"ecoinvent-{config.ecoinvent_version}-{system_model}"

    if config.source_database != expected_database_name:
        print(
            "WARNING: Your configured source database name does not match "
            "Brightway's default ecoinvent naming pattern."
        )
        print("Configured source database:", config.source_database)
        print("Expected database name:", expected_database_name)

    if config.source_database in bd.databases:
        print(f"{config.source_database} already exists in this Brightway project.")
    else:
        bi.import_ecoinvent_release(
            version=config.ecoinvent_version,
            system_model=system_model,
            username=ecoinvent_username,
            password=ecoinvent_password,
            lci=True,
            lcia=True,
            biosphere_name=config.biosphere_database,
            use_mp=False,
        )

        print(f"Ecoinvent {config.ecoinvent_version} ({system_model}) imported.")

### 4. Validate the ecoinvent activity selector CSV
Validate the CSV using:

```bash
windtrace-ecoinvent-updater validate --config windtrace_config.yml
```

In [ ]:
cutoff, new_db, biosphere3 = setup_brightway(config)

print("Source database activities:", len(cutoff))
print("Biosphere flows:", len(biosphere3))
print("Output database activities:", len(new_db))

Let's check that the databases where created properly. The source ecoinvent database and biosphere database should not be empty.

In [ ]:
assert len(cutoff) != 0
assert len(biosphere3) != 0

print("Source database activities:", len(cutoff))
print("Biosphere flows:", len(biosphere3))
print("Output database activities:", len(new_db))

### 5. Create life-cycle inventories

The function allowing to create customized inventories is **lci_wind_turbine()**, which admits the following parameters:
| Parameter                     | Description                                                                  | Default                             | Type |
|------------------------------|------------------------------------------------------------------------------|-------------------------------------|------|
| Park name                    | Name of the wind park                                                        | *Required*                          | `str` |
| Park power                   | Total power of the wind park (MW)                                            | *Required*                          | `float` |
| Number of turbines           | Number of turbines in the wind park                                          | *Required*                          | `int` |
| Park location                | Country code (ISO 3166-1 alpha-2)                                            | *Required*                          | `str` |
| Park coordinates             | Latitude and longitude (WGS84)                                               | *Required*                          | `tuple` |
| Manufacturer                 | Turbine manufacturer                                                         | `LM Wind`                           | `'LM Wind'`, `'Vestas'`, `'Siemens Gamesa'`, `'Enercon'`, `'Nordex'` |
| Rotor diameter               | Rotor diameter (m)                                                           | *Required*                          | `int` |
| Turbine rated power          | Nominal power per turbine (MW)                                               | *Required*                          | `float` |
| Hub height                   | Turbine hub height (m)                                                        | *Required*                          | `float` |
| Regression adjustment        | Steel mass as function of `D2h` or `Hub height`                              | `D2h`                               | `'D2h'`, `'Hub height'` |
| Commissioning year           | Year turbine started operation                                               | *Required*                          | `int` |
| Recycled steel share         | Recycled content share of steel                                              | Eurofer data (2012–2021)            | `float` |
| Land use permanent intensity | Permanent land transformation per MW (m²/MW)                                 | `3000`                              | `float` |
| Electricity mix steel        | Electricity mix for steel production                                         | Eurofer country mix                 | `'Norway'`, `'Europe'`, `'Poland'` *(optional)* |
| Generator type               | Gearbox and generator type                                                   | `Doubly fed induction generator`    | `'dd_eesg'`, `'dd_pmsg'`, `'gb_pmsg'`, `'gb_dfig'` *(optional)* |
| Land cover type              | Land cover before installation                                               | *Required*                          | `'crops, non-irrigated'`, `'grassland'`, `'road'`, etc. |
| End-of-life scenario         | Material recycling scenario: <br> `1`: baseline<br> `2`: optimistic<br> `3`: pessimistic | `Baseline` | `str` |
| Lifetime                     | Expected lifetime of the turbine (years)                                     | `20`                                | `int` |
| Capacity factor              | Avg. delivered electricity / theoretical max                                 | `0.24`                              | `float` |
| Attrition rate               | Annual performance degradation                                               | `0.009`                             | `float` |


Parameters without a default value must be filled. Parameters with a default value are not compulsory but can be adapted to the user needs.

### 5.1. Example of application

Let's build an imaginary 100 MW wind park ('my_wind_park') of 20 turbines, 5 MW each, in a random location in Spain. \
The manufacturer of the turbine is Siemens Gamesa, with turbines of 130 m rotor diameter, 140 m hub height and a gear-boxed double-fed induction generation drivetrain. \
The park is commissioned in 2020 and is expected to last for 25 years. \
The expected initial capacity factor is 24%. \
**IMPORTANT**: make sure the name you give to the park (park_name variable) has not been used previously in your database! This means, you cannot create two wind parks with the exact same name in the same database.

In [ ]:
lci_wind_turbine(
    park_name='my_wind_park',
    park_power=100.0,
    number_of_turbines=20,
    park_location='ES',
    park_coordinates=(41.502, -1.126),
    manufacturer='Siemens Gamesa',
    rotor_diameter=130,
    turbine_power=5.0,
    hub_height=140,
    commissioning_year=2020,
    generator_type='gb_dfig',
    lifetime=25,
    cf=0.24,

    new_db=new_db,
    biosphere3=biosphere3,
    cutoff391=cutoff,

    ecoinvent_version=config.ecoinvent_version,
    activity_variables_csv=config.activity_variables_csv,
)

If you need to delete the new database: 
WARNING: All inventories created will be deleted!!

In [ ]:
# Uncomment the below lines to delete all activities in the new database
# from windtrace.onshore import delete_db
# delete_db(config.output_database) 

### 5.2 Exploring the inventories

The inventories are given by unit of turbine, unit of park, and kWh (for both turbine and park). In this example, the inventories would have the following names (and codes):

- Turbine (FU=unit): my_wind_park_single_turbine
- Turbine (FU=kWh): my_wind_park_turbine_kwh
- Park (FU=unit): my_wind_park_100.0
- Park (FU=kWh): my_wind_park_kwh

You can access them as follows:

In [ ]:
# park (FU=unit)
park_unit_act = new_db.get('my_wind_park_100.0')
# park (FU=kWh)
park_kwh_act = new_db.get('my_wind_park_park_kwh')
# turbine (FU=unit)
turbine_unit_act = new_db.get('my_wind_park_single_turbine')
# turbine (FU=kWh)
turbine_kwh_act = new_db.get('my_wind_park_turbine_kwh')

Alternatively, if your interest is only the materials of one turbine, you can access them with the following code (and name):
- Turbine materials: my_wind_park_materials

In [ ]:
turbine_act = new_db.get('my_wind_park_materials')
for ex in turbine_act.technosphere():
    print(f'{ex.input["name"]}: {ex["amount"]} ({ex.unit})')
# Note that steel and chromium steel will be both printed twice. The first appearance corresponds to the turbine, and the second one to the foundations.

### 5.3 Calculating LCIA scores

LCIA scores can be calculated with a conventional LCA brightway approach.

Notes: 
1. The LCIA method names must exist in your Brightway project. If a method is missing, check `bd.methods` and choose an available method.
2. Some LCIA method tuples differ across Brightway/ecoinvent installations. If the output keys from `lca_wind_turbine()` are not displayed as expected, make sure to change 'ecoinvent-3.9.1' in method/indicator name to your current ecoinvent version.

In [ ]:
# wind park (total climate change impact)
park_total = new_db.get('my_wind_park_100.0')
lca_obj = park_total.lca(amount=1, method=('ecoinvent-3.12', 'EF v3.1', 'climate change', 'global warming potential (GWP100)'))
score = lca_obj.score
print(f'LCIA score for FU = one wind park {score}')

# wind park (climate change per kWh)
park_kwh = new_db.get('my_wind_park_park_kwh')
lca_obj = park_kwh.lca(amount=1, method=('ecoinvent-3.12', 'EF v3.1', 'climate change', 'global warming potential (GWP100)'))
score = lca_obj.score
print(f'LCIA score for FU = one kWh (park) {score}')

# wind turbine (total climate change impact)
turbine_total = new_db.get('my_wind_park_single_turbine')
lca_obj = turbine_total.lca(amount=1, method=('ecoinvent-3.12', 'EF v3.1', 'climate change', 'global warming potential (GWP100)'))
score = lca_obj.score
print(f'LCIA score for FU = one turbine {score}')

# wind turbine (climate change per kWh)
turbine_kwh = new_db.get('my_wind_park_turbine_kwh')
lca_obj = turbine_kwh.lca(amount=1, method=('ecoinvent-3.12', 'EF v3.1', 'climate change', 'global warming potential (GWP100)'))
score = lca_obj.score
print(f'LCIA score for FU = one kWh (turbine) {score}')


### or multi-lca! [example: wind turbine]
# create a list with EF v3.1 methods
ef_v3_1 = [m for m in bd.methods if m[0] == 'EF v3.1']
# set activities
turbine_total = new_db.get('my_wind_park_single_turbine')
turbine_kwh = new_db.get('my_wind_park_turbine_kwh')
# create empty dict where results will be saved
results = {}
# calculate impacts
for act in [turbine_total, turbine_kwh]:
    results[str(act)] = {}
    lca_obj = act.lca(amount=1)
    for m in ef_v3_1:
        lca_obj.switch_method(m)
        lca_obj.lcia()
        results[str(act)][str(m)] = lca_obj.score


Alternatively, LCA impacts of a park or an individual turbine of the park can be calculated with lca_wind_turbine(). It returns two dictionaries: the first one being results per unit and the second one being results per kWh. Default results are given per turbine. For the whole park, you must set the parameter 'turbine' to False. ReCiPe 2016 v1.03, midpoint (H) is the default LCIA method. It can be manually changed by giving the parameter 'method' another name of a method in Brightway.

Here an example of application for the whole park. The first dictionary saves scores per FU=unit, and the second per FU=1kWh

In [ ]:
lcia_scores_unit, lcia_scores_kwh = lca_wind_turbine(park_name='my_wind_park', park_power=100.0, method='EF v3.1', turbine=False, new_db=new_db)
print(f'LCIA scores for (FU = one wind park unit) {lcia_scores_unit}')
print(f'LCIA scores for (FU = one kwh) {lcia_scores_kwh}')

You can also give it a certain set of indicators of your choice as a list with the variable 'indicators'. Example:

In [ ]:
lcia_scores_unit, lcia_scores_kwh = lca_wind_turbine(park_name='my_wind_park', park_power=100.0, method='EF v3.1',
                               indicators=[('ecoinvent-3.12', 'CML v4.8 2016 no LT', 'acidification no LT', 'acidification (incl. fate, average Europe total, A&B) no LT'),
                                           ('ecoinvent-3.12', 'Cumulative Energy Demand (CED)', 'energy resources: renewable', 'energy content (HHV)'),
                                           ('ecoinvent-3.12', 'Ecological Scarcity 2021 no LT', 'climate change no LT', 'global warming potential (GWP100) no LT')
                                          ],
                               turbine=False, new_db=new_db)
print(f'LCIA scores for (FU = one wind park unit) {lcia_scores_unit}')
print(f'LCIA scores for (FU = one kwh) {lcia_scores_kwh}')

### 5.4 Another example with land use focus

Now let's build another wind park changing a bit the land use related parameters. Let's say we know the original land where the turbine will be built is a pasture. Let's say we also know that the permanent land use intensity is smaller than the default number we use. This parameter should include the squared meters per MW of land that will be occupied by permanent access roads, transformer, substation and the turbines themselves. We will use an intensity of 1000 m2/MW this time.

In [ ]:
lci_wind_turbine(
    park_name='my_wind_park_LandUse',
    park_power=100.0,
    number_of_turbines=20,
    park_location='ES',
    park_coordinates=(41.502, -1.126),
    manufacturer='Siemens Gamesa',
    rotor_diameter=130,
    turbine_power=5.0,
    hub_height=140,
    commissioning_year=2020,
    generator_type='gb_dfig',
    lifetime=25,
    cf=0.24,
    land_use_permanent_intensity=1000,
    land_cover_type='pasture',

    new_db=new_db,
    biosphere3=biosphere3,
    cutoff391=cutoff,

    ecoinvent_version=config.ecoinvent_version,
    activity_variables_csv=config.activity_variables_csv,
)

And compare the land-use impacts with the previous wind park we created:

In [ ]:
previous_park_score_unit, previous_park_score_kwh = lca_wind_turbine(park_name='my_wind_park', park_power=100.0, method='EF v3.1', 
                               indicators=[('ecoinvent-3.12', 'EF v3.1', 'land use', 'soil quality index')
                                          ], 
                               turbine=False, new_db=new_db)
new_park_score_unit, new_park_score_kwh  = lca_wind_turbine(park_name='my_wind_park_LandUse', park_power=100.0, method='EF v3.1', 
                               indicators=[('ecoinvent-3.12', 'EF v3.1', 'land use', 'soil quality index')
                                          ], 
                               turbine=False, new_db=new_db)
print(f'Land Use score previous park (FU=one park): {previous_park_score_unit}; (FU=1kWh): {previous_park_score_kwh}')
print(f'Land Use score new park(FU=one park): {new_park_score_unit}; (FU=1kWh): {new_park_score_kwh}')

### 5.5 Creating a future wind park

There are some input variables that allow to model more 'futuristic' wind turbine designs. For instance, if the turbine you are designing is going to be installed 10 years from now, you may want to consider that the steel industry (with a huge impact in wind turbines) has improved its recycling shares, and is using a cleaner electricity mix. Moreover, you can adapt the end-of-life scenario to consider optimistic recycling rates. Finally, you may want to add a higher capacity factor (due to, for instance, technical improvements). A summary of how would this be implemented:
- Steel recycling rate improvement: recycled_share_steel=0.6 (European recycled steel share in 2021 was 0.435)
- Steel electricity mix improvement: electricity_mix_steel='Norway' (only options: 'Norway', 'Europe', 'Poland')
- Optimistic recycling rates (rr): eol_scenario=2 (eol_scenario=1 means current rr, eol_scenario=2 means optimistic rr, eol_scenario=3 means pessimistic rr)
- Higher capacity factor: cf=0.4 (This is a 40%. The European mean in 2024 was 24%)

In [ ]:
lci_wind_turbine(
    park_name='my_wind_park_future',
    park_power=100.0,
    number_of_turbines=20,
    park_location='ES',
    park_coordinates=(41.502, -1.126),
    manufacturer='Siemens Gamesa',
    rotor_diameter=130,
    turbine_power=5.0,
    hub_height=140,
    commissioning_year=2030,
    generator_type='gb_dfig',
    lifetime=25,
    cf=0.40,
    eol_scenario=2,
    electricity_mix_steel='Norway',
    recycled_share_steel=0.6,

    new_db=new_db,
    biosphere3=biosphere3,
    cutoff391=cutoff,

    ecoinvent_version=config.ecoinvent_version,
    activity_variables_csv=config.activity_variables_csv,
)

And compare climate change impacts with the previous turbine modelled in section 5.1 in this notebook:

In [ ]:
previous_park_score_unit, previous_park_score_kwh = lca_wind_turbine(park_name='my_wind_park', park_power=100.0, method='EF v3.1',
                               indicators=[('ecoinvent-3.12', 'EF v3.1', 'climate change', 'global warming potential (GWP100)')
                                          ],
                               turbine=False, new_db=new_db)
new_park_score_unit, new_park_score_kwh  = lca_wind_turbine(park_name='my_wind_park_future', park_power=100.0, method='EF v3.1',
                               indicators=[('ecoinvent-3.12', 'EF v3.1', 'climate change', 'global warming potential (GWP100)')
                                          ],
                               turbine=False, new_db=new_db)
print(f'Climate Change score previous park (FU=one park): {previous_park_score_unit}; (FU=1kWh): {previous_park_score_kwh}')
print(f'Climate Change score new park(FU=one park): {new_park_score_unit}; (FU=1kWh): {new_park_score_kwh}')